In [ ]:
%pip install -q openai python-dotenv pillow

In [ ]:
import os
import io
import json
import base64
from pathlib import Path
from dotenv import load_dotenv
from PIL import Image
from openai import OpenAI

load_dotenv(override=True, dotenv_path="../.env.local")
api_key = os.getenv("OPENAI_API_KEY")
if not api_key:
    raise ValueError("OPENAI_API_KEY is missing in ../.env.local")

client = OpenAI(api_key=api_key)
print(f"API Key loaded: {api_key[:4]}...{api_key[-4:]}")

In [ ]:
# Cabinet image list
image_paths = [
    Path("data/2026-02-16 13.34.42.jpg"),
    Path("data/2026-02-16 13.34.48.jpg"),
    Path("data/2026-02-16 13.35.02.jpg"),
]

for p in image_paths:
    if not p.exists():
        raise FileNotFoundError(f"Missing image: {p}")

def optimize_image_to_base64(image_path: Path, max_side: int = 1024, quality: int = 60):
    """Resize/compress image to reduce token cost before sending."""
    img = Image.open(image_path).convert("RGB")
    original_w, original_h = img.size

    # Keep aspect ratio and reduce large images
    img.thumbnail((max_side, max_side))
    optimized_w, optimized_h = img.size

    buf = io.BytesIO()
    img.save(buf, format="JPEG", quality=quality, optimize=True)
    optimized_bytes = buf.getvalue()

    return {
        "b64": base64.b64encode(optimized_bytes).decode("utf-8"),
        "mime": "image/jpeg",
        "original_size": (original_w, original_h),
        "optimized_size": (optimized_w, optimized_h),
        "optimized_kb": round(len(optimized_bytes) / 1024, 1),
    }

In [ ]:
def extract_json(text: str) -> dict:
    start = text.find("{")
    end = text.rfind("}")
    if start == -1 or end == -1 or end <= start:
        return {"raw_response": text.strip()}
    try:
        return json.loads(text[start:end+1])
    except json.JSONDecodeError:
        return {"raw_response": text.strip()}

results = []
for image_path in image_paths:
    payload = optimize_image_to_base64(image_path, max_side=1024, quality=60)

    response = client.chat.completions.create(
        model="gpt-5-nano",
        messages=[
            {
                "role": "system",
                "content": "You are a cabinet quality inspector. Return only valid JSON."
            },
            {
                "role": "user",
                "content": [
                    {
                        "type": "text",
                        "text": "Inspect this cabinet image and return JSON with keys: defect_type, defect_location, defect_description, severity, suggested_action."
                    },
                    {
                        "type": "image_url",
                        "image_url": {
                            "url": f"data:{payload['mime']};base64,{payload['b64']}",
                            "detail": "low"
                        }
                    }
                ]
            }
        ],
    )

    content = response.choices[0].message.content or ""
    data = extract_json(content)

    results.append({
        "image": image_path.name,
        "original_size": f"{payload['original_size'][0]}x{payload['original_size'][1]}",
        "optimized_size": f"{payload['optimized_size'][0]}x{payload['optimized_size'][1]}",
        "optimized_kb": payload['optimized_kb'],
        "defect_type": data.get("defect_type", "N/A"),
        "defect_location": data.get("defect_location", "N/A"),
        "severity": data.get("severity", "N/A"),
        "suggested_action": data.get("suggested_action", "N/A"),
    })

results

In [ ]:
# Print a compact table
headers = ["image", "original_size", "optimized_size", "optimized_kb", "defect_type", "defect_location", "severity"]
width = {h: max(len(h), *(len(str(r[h])) for r in results)) for h in headers}

line = " | ".join(h.ljust(width[h]) for h in headers)
sep = "-+-".join("-" * width[h] for h in headers)
print(line)
print(sep)
for r in results:
    print(" | ".join(str(r[h]).ljust(width[h]) for h in headers))
